# Implementation of a Neural Network for Iris dataset classification. Part 1: Data preparation

This notebook is prepares the data from the Iris Dataset to be used to train a multilayer neural network (implemented in another workbook) for classifying flower species in the Iris dataset.

The code is separated in Data preparation and Network training to facilitate changing the modelling step


1. [Introduction](#Introduction)
2. [Data Loading](#Data-Loading)
3. [Data preparation](#Exploratory-Data-Analysis)



## Introduction <a name="introduction"></a>

In this lab we build a simple N, implementing every fuunction from scratch.

The problem to be solved is the identification (*classification*, *prediction*) of the species of an Iris flower using the diemnsuions of the petals and the sepals. The `Iris` dataset we will use was introduced by R.A. Fisher to illustrate his works on Linear Discriminant Analysis.

We will follow a sequence of steps that align well with the process of building and validating a ML model. 

Many of the functions are available in high level libraries -like Scikit-Learn or Caret- but, for illustration purposes they will be written from scratch.


## Importing libraries

In [6]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


## Load and preprocess data

### Reading the data

In [8]:
#read data from csv
iris = pd.read_csv("data/IrisData1.csv")
selected_rows = [0, 1, 50, 51, 100, 101]
print(iris.iloc[selected_rows])


     Sepal_Length  Sepal_Width  Petal_Length  Petal_Width          Species
0             5.1          3.5           1.4          0.2      Iris-setosa
1             4.9          3.0           1.4          0.2      Iris-setosa
50            7.0          3.2           4.7          1.4  Iris-versicolor
51            6.4          3.2           4.5          1.5  Iris-versicolor
100           6.3          3.3           6.0          2.5   Iris-virginica
101           5.8          2.7           5.1          1.9   Iris-virginica


### Dataset verification

A standard step in the Data Analysis pipeline is to check the data, in order to detect possible wrong or problematic values such as missing data, outliers, mistakes (ex. negative length) etc.

While this is very context dependent a function to automate the checking is defined and applied below.

In [18]:
#Check the dataset to make sure no data is missing and Check the class labels
def verify_dataset(data):
  #if any of the rows have missing value return datas missing
  data_found = 1
  for each_column in data.columns:
    if data[each_column].isnull().any():
            print("Data missing in Column " + each_column)
            #if any rows are not missing return Dataset is complete. No missing value
            quit()
    if data_found == 1:
        print("Dataset is complete. No missing values. Ok")
    return

verify_dataset(iris)

Dataset is complete. No missing values. Ok


### One hot encoding function

In the **Iris dataset**, the target variable `Species` consists of three categorical classes:

- *Iris-setosa*
- *Iris-versicolor*
- *Iris-virginica*

Machine learning models, especially neural networks, require numerical inputs. However, categorical values like `"Iris-setosa"` cannot be directly used in mathematical computations. In cases like this a common strategy is tio use *one_hot encoding* 

One-hot encoding is a technique used to convert categorical labels into a numerical format that can be processed by machine learning models. In this method, each category is represented as a binary vector where only one element is `1`, and all others are `0`.

One-hot encoding transforms them into numerical representations like:

| Species          | One-Hot Encoding |
|------------------|-----------------|
| Iris-setosa     | `[1, 0, 0]`       |
| Iris-versicolor | `[0, 1, 0]`       |
| Iris-virginica  | `[0, 0, 1]`       |

This ensures that the model **does not assign an ordinal relationship** between categories (e.g., assuming *Iris-virginica* > *Iris-setosa* if labeled as 0,1,2).


In [27]:
#This function accepts an array of categorical variables and returns the one hot encoding
def to_one_hot(Y):
    n_col = np.amax(Y) + 1 # Determine the number of unique classes
    binarized = np.zeros((len(Y), n_col)) # Create a matrix filled with zeros
    for i in range(len(Y)):
        binarized[i, Y[i]] = 1.
    return binarized


### Data normalization function

The training of many ML models, like ANNs, can be negatively affected by input features with different scales. Some of the problems this can yield are:
- Features with larger numerical ranges can dominate smaller ones, leading to biased weight updates in the model.
- Gradient-based optimization methods (like *gradient descent*) converge faster when input features are on a similar scale.
- Large values may lead to computational issues such as *vanishing or exploding gradients*.

This is typically achieved by transforming the data so that all features have comparable scales, ensuring they contribute equally to a machine learning model. Normalization can be applied column-wise (scaling each feature independently) to preserve relationships among variables, or row-wise (scaling each observation to unit length), which is useful when working with directional vectors.

THe original version of this lab provided a row-wise normalization that puts all values on the same scale, but affects the relation among variables. Instead, we are going to use *column-wise normalization*

In this case, we apply column-wise normalization because the features represent absolute measurements of floral organs, not unit vectors or directional data. This ensures that the relationships among features are maintained while preventing differences in scale from affecting the learning process.

In [35]:
# Row-wise normalization function. Removed
# def normalize(X, axis=-1, order=2):
#    l2 = np.atleast_1d(np.linalg.norm(X, order, axis))
#    l2[l2 == 0] = 1
#    return X / np.expand_dims(l2, axis)

# Column-wise normalization
def normalize(X, axis=0, order=2):
    # Compute the L2 norm for each column (axis=0)
    l2 = np.atleast_1d(np.linalg.norm(X, order, axis))  
    # Prevent division by zero by setting zero norms to 1
    l2[l2 == 0] = 1 
    # Normalize each column dividing it by its respective L2 norm
    return X / np.expand_dims(l2, axis)

### Running the preprocessing steps

#### Selecting features and transforming dataframe into aray

In [42]:
# Subset the dataset keeping only numeric features
selected_features = {
    "Sepal_Length": True,
    "Sepal_Width": True,
    "Petal_Length": True,
    "Petal_Width": True
}

# Filtrar solo las columnas seleccionadas
x = iris.loc[:, [col for col, include in selected_features.items() if include]]
x = x.to_numpy() # Change data from Pandas Dataframe to numpy array

#### Applying one hot encoding

In [48]:
#Replace the species with 1,2 or 3 as appropriate
label_dict = dict()
label_dict['0'] = 'setosa'
label_dict['1'] = 'virginica'
label_dict['2'] = 'versicolor'
iris['Species'].replace(['Iris-setosa', 'Iris-virginica', 'Iris-versicolor'], [0, 1, 2], inplace=True)

#Get Output, flatten and encode to one-hot
columns = ['Species']

y = pd.DataFrame(iris, columns=columns)
y = y.to_numpy()
y = y.flatten()
y = to_one_hot(y)
print(y[selected_rows])


[[1. 0. 0.]
 [1. 0. 0.]
 [0. 0. 1.]
 [0. 0. 1.]
 [0. 1. 0.]
 [0. 1. 0.]]


## Prepare Data for Training

Before we can train the model we will need to split the data in train and tests

Data normalization is performed columnwise, so it has to be done after train/test split.

Last, we will define all the functions needed for traing, from activation functions to forward and backpropagation calculations.

### Test-Training split function

In [53]:
x_y = pd.DataFrame(np.concatenate((x,y), axis=1))
def split_dataset_test_train(data,train_size):
  data = data.sample(frac=1).reset_index(drop=True)
  training_data = data.iloc[:int(train_size * len(data))].reset_index(drop=True)
  testing_data = data.iloc[int(train_size * len(data)):].reset_index(drop=True)
  return [training_data, testing_data]

#### Split the data

In [57]:
train_test_data = split_dataset_test_train(x_y,0.7)
X_train = train_test_data[0].iloc[:,0:4].to_numpy()
X_test = train_test_data[1].iloc[:,0:4].to_numpy()

y_train = train_test_data[0].iloc[:,-3:].to_numpy()
y_test = train_test_data[1].iloc[:,-3:].to_numpy()

#### Normalize test and train sets separately

In [60]:
X_test = normalize(X_test)
y_test = normalize(y_test)

### Save the data for later use

In [62]:
# Save data into a binary file
np.savez("iris_train_test_data.npz", X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test)

print("Dataset saved successfully as 'iris_train_test_data.npz'.")


Dataset saved successfully as 'iris_train_test_data.npz'.
